# Fast ML UK tutorial

# Part 1: Introduction
Welcome to the Fast ML UK tutorial at Imperial College London.

In this notebook, we will show how to use the hls4ml library to convert a Keras neural network model into an FPGA implementation using Intel's oneAPI toolkit. 

This will be done in the context of a realistic LHC dataset composed of **signal events: di-Higgs production**:

and **background events** from several high cross-section processes:
* **QCD multijet production**
* **Drell-Yan production**: with additional jets
* **W boson production**: with additional jets

Our goal is to train a machine learning (ML) model to classify each event into one of these four processes using the reconstructed particles in that event. You are provided with a labelled dataset from simulation containing roughly equal numbers of each of the four processes. Each event is represented by an array of the 32 most energetic particles in that event. Each particle comes with a set of three kinematic features:
* Transverse momentum ($p_T$)
* Pseudorapidity ($\eta$)
* Azimuthal angle ($\phi$) 

Note this is a simplified dataset, and in real life we would also have other features such as particle ID, charge, etc to help with the classification task.

The ultimate aim is to "accelerate" the inference of this ML model on an FPGA, so that it can be used in the trigger system of an LHC experiment to make real-time decisions about which events to keep for further analysis.

We will begin by training a simple multi-layer perceptron (MLP) for this task, which uses flattened inputs for all particles in an event. As an extension to this notebook, we will explore the application of the DeepSet architecture and compare its performance and FPGA implementation to the MLP.

## 1a: Understanding the dataset
In this section we will import the libraries, define the data loading functions, and load the dataset. Following this we will take some time to understand the structure of the data and the features that we have available for training our ML model.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

import keras
from keras import layers, models, regularizers
from keras.optimizers import Adam
from keras.optimizers.schedules import CosineDecay
from keras.callbacks import ModelCheckpoint

import hls4ml

from matplotlib import pyplot as plt

# Define any common variables
label_map = {
    'QCD': 0,
    'HH4b': 1,
    'Drell-Yan': 2,
    'W+Jets': 3
}

color_map = {
    'QCD': 'blue',
    'HH4b': 'orange',
    'Drell-Yan': 'green',
    'W+Jets': 'red'
}

# Function to prepare the dataset for training and testing
# This will also apply a per-feature normalization which is crucial for training neural networks effectively.
def prep_physics_data(X, y, num_particles=10, test_size=0.2, random_state=42):
    # 1. Select the Top K most energetic particles
    # Since particles are already sorted descending by pT, we just slice the array
    X = X[:, :num_particles, :]
    
    # 2. Scramble (Shuffle) the dataset
    np.random.seed(random_state)
    indices = np.random.permutation(len(X))
    X = X[indices]
    y = y[indices]
    
    # 3. Train-Test Split
    split_idx = int(len(X) * (1 - test_size))
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    
    # 4. Per-Feature Normalization (Safeguarded against padding)
    # We create a mask identifying where the real particles are (e.g., pT != 0)
    train_mask = (X_train[..., 0] != 0)[..., np.newaxis] 
    test_mask = (X_test[..., 0] != 0)[..., np.newaxis]

    # Calculate mean and std ONLY using the training set to prevent data leakage.
    # We take the mean across the batch (0) and particle (1) dimensions.
    mean = np.mean(X_train, axis=(0, 1), keepdims=True)
    std = np.std(X_train, axis=(0, 1), keepdims=True)
    
    # Prevent division by zero just in case a feature is totally empty
    std = np.where(std == 0, 1e-7, std)
    
    # Normalize
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std
    
    # RE-MASK: Force the padded particles back to exactly 0.0
    X_train = X_train * train_mask
    X_test = X_test * test_mask
    
    return X_train, X_test, y_train, y_test


# Custom DataLoader for Keras that handles our 3D particle data and supports optional flattening
class ParticleDataLoader(keras.utils.PyDataset):
    def __init__(self, X_data, y_data, batch_size=256, flatten=False, **kwargs):
        super().__init__(**kwargs)
        
        # Flatten the entire dataset once during initialization
        if flatten:
            # Reshape from (Total_Events, num_particles, features) 
            # to (Total_Events, num_particles * features)
            self.X_data = X_data.reshape(X_data.shape[0], -1)
        else:
            self.X_data = X_data
            
        self.y_data = y_data
        self.batch_size = batch_size

    def __len__(self):
        # Defines the number of batches per epoch
        return int(np.ceil(len(self.X_data) / self.batch_size))

    def __getitem__(self, idx):
        # Fetch the batch slice
        start_idx = idx * self.batch_size
        end_idx = start_idx + self.batch_size
        
        # The data is already in the correct shape, so we just slice it
        batch_x = self.X_data[start_idx:end_idx]
        batch_y = self.y_data[start_idx:end_idx]
        
        return batch_x, batch_y
    

# Patch for hls4ml for nicely formatting the model summary
def patched_make_html_table_template(table_header, row_templates):
    """Hotfix for broken table creation"""
    num_columns = len(next(iter(row_templates.values())))

    _row_html_template = '        <tr><td>{}</td>' + ''.join('<td>{{{}}}</td>' for _ in range(num_columns)) + '</tr>'

    table_rows = '\n'.join(
        [_row_html_template.format(row_title, *row_keys) for row_title, row_keys in row_templates.items()]
    )
    return hls4ml.report.oneapi_report._table_base_template.format(
        colspan=num_columns + 1, 
        table_header=table_header, 
        table_rows=table_rows
    )

In [ ]:
# Load the dataset
X = np.load("data/Collide_X.npy")
y = np.load("data/Collide_y.npy")

print(f"Original shapes: X={X.shape}, y={y.shape}")
print(f"Unique labels in y: {np.unique(y)} = {[list(label_map.keys())[list(label_map.values()).index(i)] for i in np.unique(y)]}")

Let's have a look at an event from each class to get a feel for the data.

The following function will plot the event display (in 3D space) i.e the trajectories of the 32 most energetic particles in an event. The particle with highest transverse momentum (pT) is labelled. We use a different colour for each class of event.

We have done the (hard) work and found events in the dataset which are representative of the different classes.

In [ ]:
def plot_event_display(x, label='QCD', num_particles=-1):
    pt = x[:, 0]
    eta = x[:, 1]
    phi = x[:, 2]
    d0 = np.zeros_like(pt)  # Placeholder for d0, since we don't have it in the dataset

    idx = np.argsort(pt)[::-1]  # Sort particles by pT in descending order and show only the top num_particles
    if num_particles > 0:
        idx = idx[:num_particles]
    pt = pt[idx]
    eta = eta[idx]
    phi = phi[idx]
    
    R = 1.0
    d0_scale = 0.1

    x_end = np.sinh(eta)
    y_start = d0_scale * np.cos(phi)
    z_start = d0_scale * np.sin(phi)
    y_end = y_start + R * np.cos(phi)
    z_end = z_start + R * np.sin(phi)

    fig = plt.figure(figsize=(7, 5), dpi=100)
    ax = fig.add_subplot(projection='3d')

    phi_grid = np.linspace(0, 2 * np.pi, 100)
    x_grid = np.linspace(-5, 5, 100)
    Phi, Xg = np.meshgrid(phi_grid, x_grid)
    Yc = R * np.cos(Phi)
    Zc = R * np.sin(Phi)
    ax.plot_surface(Xg, Yc, Zc, alpha=0.1)

    # Plot particles
    for i, (xs, ys, zs, xe, ye, ze, et, ph, p) in enumerate(zip(np.zeros_like(x_end), y_start, z_start, x_end, y_end, z_end, eta, phi, pt)):
        # Double thickness for highest pT particle
        if i == 0:
            linewidth = 2
        else:
            linewidth = 0.5
        ax.plot([xs,max(min(xe,5),-5)], [ys,ye], [zs,ze], color=color_map[label], alpha=0.5, linewidth=linewidth)
        if (i == 0):
            ax.text(max(min(xe,5),-5), ye, ze, f"[$p_T$={p:.1f} GeV, $\\eta$={et:.2f}, $\\phi$={ph:.2f}]", fontsize=8)

    # Add primary interaction point
    ax.scatter(0, 0, 0, color='black', s=1)
    ax.text(0, 0, 0, "IP", fontsize=12, ha='left', va='bottom')
    ax.set_xlim(-5,5)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    ax.set_xlabel("z ~ sinh(η)")
    ax.set_box_aspect((2,1,1))  # Aspect ratio so that x-axis is twice as long as y and z
    ax.set_title(f"Event Display for {label} event")

    plt.tight_layout()
    plt.show()

In [ ]:
# Index of representative events for each class
events_to_plot = {
    'QCD': 49610,
    'HH4b': 107498,
    'Drell-Yan': 184356,
    'W+Jets': 294899
}

for label, idx in events_to_plot.items():
    plot_event_display(X[idx], label=label, num_particles=32)


We can also look at 2D projection in eta-phi space, where the size of the marker is proportional to pT. This is a common way to visualise events in particle physics.

In [ ]:
# For each event plot particles in eta-phi space, where size of marker is proportional to pT
fig, ax = plt.subplots(2, 2, figsize=(8, 8))
for i, (label, idx) in enumerate(events_to_plot.items()):
    row_idx = i // 2
    col_idx = i % 2
    
    # Extract the original unnormalized features for the event
    event_features = X[idx]
    pT = event_features[:, 0]
    eta = event_features[:, 1]
    phi = event_features[:, 2]
    
    # Plot eta-phi with marker size proportional to pT
    sc = ax[row_idx, col_idx].scatter(phi, eta, s=10*pT, alpha=0.5, color=color_map[label])
    ax[row_idx, col_idx].set_title(f"Event Index: {idx} | True Label: {label}")
    ax[row_idx, col_idx].set_xlabel("Phi")
    ax[row_idx, col_idx].set_ylabel("Eta")
    ax[row_idx, col_idx].set_xlim(-np.pi, np.pi)
    ax[row_idx, col_idx].set_ylim(-5, 5)
    ax[row_idx, col_idx].grid()
plt.tight_layout()
plt.show()

Can you tell these events apart by eye?

Remember this is just one event from each class. We will train a ML model to learn the different particle kinematic patterns in each class, so that it can classify new events that it has never seen before.

Let's briefly look at the distributions over the whole dataset e.g. for the 3 most energetic particles in each event.

Of course, an ML model is not limited to 1D histograms, but can learn complex correlation between the particles. This is crucial for distinguishing between the different classes.

In [ ]:
# Plot pT, eta and phi distributions for the five most energetic particles in each event, separated by class (5 x 3 plots)
num_particles_to_plot = 3
features_to_plot = [0, 1, 2]  # pT, eta, phi
feature_names = ['pT', 'eta', 'phi']
range_map = {
    0: (0, 100),   # pT range
    1: (-5, 5),    # eta range
    2: (-np.pi, np.pi)  # phi range
}

fig, axs = plt.subplots(len(features_to_plot), num_particles_to_plot, figsize=(12, 12))
for i, feature_idx in enumerate(features_to_plot):
    for j in range(num_particles_to_plot):
        ax = axs[i, j]
        for label, label_idx in label_map.items():
            # Extract the feature values for the j-th particle and the current feature index, for events of the current class
            feature_values = X[y == label_idx, j, feature_idx]
            ax.hist(feature_values, bins=50, range=range_map[feature_idx], alpha=0.5, histtype='step', label=label, density=True, color=color_map[label])
        ax.set_title(f"Particle {j+1}")
        ax.set_xlabel(feature_names[i])
        ax.set_ylabel("Density")
        ax.legend()
plt.tight_layout()
plt.show()

## 1b: Train neural network for event classification
We are going to train a simple multi-layer perceptron (MLP) to classify between the four processes in our dataset. First, we need to do a bit of pre-processing of the data to get it into a format that can be used for training. We will then define the architecture of our MLP and train it on the particle data.

In [ ]:
# Build the datasets for training and testing
# We will use the top 16 most energetic particles to reduce complexity. You can explore changing this and seeing how it affects the performance of your model later
num_particles = 16
X_train, X_test, y_train, y_test = prep_physics_data(X, y, num_particles=num_particles, test_size=0.2, random_state=42)
print(f"Train Shape: {X_train.shape} | Test Shape: {X_test.shape}")

In [ ]:
# Initialise the data loaders
# We will flatten the data for the MLP model (note for more complex architectures like DeepSet you may want to keep the 3D structure)
flatten = True

# We will also use a batch size of 512
train_loader = ParticleDataLoader(X_train, y_train, batch_size=512, flatten=flatten)
test_loader = ParticleDataLoader(X_test, y_test, batch_size=512, flatten=flatten)
print(f"DataLoader gives input shape: {train_loader[0][0].shape} and labels shape: {train_loader[0][1].shape}")

Now let's build a classifier. We will use a very simple MLP with three hidden layers and 32 neurons in each layer. We will use the ReLU activation function for the hidden layers and softmax for the output layer, since this is a multi-class classification problem. We will also use dropout and an L2 regularizer to prevent overfitting.

In [ ]:
# Let's build the classifier
input_shape = train_loader[0][0].shape[1]  # This will be num_particles * features after flattening = 16 * 3 = 48
inputs = layers.Input(shape=(input_shape,))

# Our network will have three hidden layers with 32 neurons each, ReLU activation, L2 regularization and dropout
x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(inputs)
x = layers.Dropout(0.1)(x)
x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(len(label_map), activation='softmax')(x)
model = models.Model(inputs=inputs, outputs=outputs)


In [ ]:
# Build optimizer with a learning rate scheduler (cosine decay)
# We will train for 20 epochs with an initial learning rate of 1e-3, decaying down to 1e-4 at the end of training.
N_epochs = 20
initial_learning_rate = 1e-3
steps_per_epoch = len(train_loader)
lr_scheduler = CosineDecay(
    initial_learning_rate=initial_learning_rate,
    decay_steps=N_epochs * steps_per_epoch,
    alpha=1e-4
)
optimizer = Adam(learning_rate=lr_scheduler)

In [ ]:
# Compile the model with the optimizer, categorical cross-entropy loss and accuracy metric
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

A very simple architecture with only 3812 trainable parameters...

In [ ]:
# Model training

# Add a ModelCheckpoint callback to save the best model based on validation accuracy during training
os.system("mkdir -p models")  # Create the models directory if it doesn't exist
checkpoint_path = "models/mlp_best_model.keras"
checkpoint_cb = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

history = model.fit(
    train_loader,
    validation_data=test_loader,
    epochs=N_epochs,
    callbacks=[checkpoint_cb]
)

train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]
print(f"Final Train Accuracy: {train_acc:.3f}")
print(f"Final Validation Accuracy: {val_acc:.3f}")
print(f"Best model saved at: {checkpoint_path}")

In [ ]:
# Plot training history: loss (left hand panel) and accuracy (right hand panel) curves
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(history.history['loss'], label='Train Loss')
ax[0].plot(history.history['val_loss'], label='Test Loss')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Loss')
ax[0].legend()
ax[1].plot(history.history['accuracy'], label='Train Accuracy')
ax[1].plot(history.history['val_accuracy'], label='Test Accuracy')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Accuracy')
ax[1].legend()
plt.tight_layout()
plt.show()

Our model seems to have trained succesfully, reaching a test accuracy of around 60%.

### Evaluate the performance of the model
Our overall metric for evaluating the performance will be the accuracy on the test set i.e. the fraction of total events that are correctly classified.

However, we can go further. In the next few cells we will first calculate the accuracy and then explore:
* Probability distributions for each class
* ROC curves and AUC for each class

In [ ]:
# Load the best model from training and evaluate it on the test set
model = keras.models.load_model(checkpoint_path)

In [ ]:
# Evaluate the accuracy of the model on the test set
y_pred_probs = model.predict(test_loader)
y_pred = np.argmax(y_pred_probs, axis=1)
test_acc = np.mean(y_pred == y_test)
print(f"Test Accuracy: {test_acc:.3f}")

In [ ]:
# Plot output probability histograms for each class, separated by true class
fig, axs = plt.subplots(2, 2, figsize=(8, 8))
for pred, pred_idx in label_map.items():

    # Extract axis indices
    row_idx = pred_idx // 2
    col_idx = pred_idx % 2
    
    for truth, truth_idx in label_map.items():
        class_mask = (y_test == truth_idx)
        axs[row_idx, col_idx].hist(y_pred_probs[class_mask, pred_idx], bins=50, range=(0,1), histtype='step', label=f"{truth}", density=True, color=color_map[list(label_map.keys())[list(label_map.values()).index(truth_idx)]])
    axs[row_idx, col_idx].set_xlabel(f"Predicted probability for class: {pred}")
    axs[row_idx, col_idx].set_ylabel("Density")
    axs[row_idx, col_idx].legend(title="True Class")
    axs[row_idx, col_idx].set_xlim(0, 1)
plt.tight_layout()
plt.show()

Do you understand what these plots are showing?

Now let's look at the ROC curves for each class. We obtain these by treating each class as "signal" and the other classes as "background", and then scanning over the threshold on the predicted probability to obtain the true positive rate and false positive rate at each threshold.

The ROC curves are evaluated for each class vs rest (black lines) and also for each pair of classes (coloured lines). The AUC is a single number that summarises the performance of the classifier for each class, with 1 being perfect classification and 0.5 being random guessing. The threshold probabilities are shown along the curve for the class vs rest case, where you can see the trade-off between true positive rate and false positive rate as you change the threshold.

In [ ]:
# Plot ROC curves for each class
fig, axs = plt.subplots(2, 2, figsize=(8, 8))

for truth, truth_idx in label_map.items():

    i = int(np.floor(truth_idx / 2))
    j = int(truth_idx % 2)

    y_true = (y_test == truth_idx).astype(int)  # Binary labels for the current class
    y_pred_prob = y_pred_probs[:, truth_idx]  # Predicted probabilities for the current class

    fpr, tpr, thresholds = roc_curve(y_true, y_pred_prob)
    auc = roc_auc_score(y_true, y_pred_prob)
    axs[i, j].plot(fpr, tpr, label=f"All others (AUC = {auc:.2f})", color='black')
    for k in range(1, len(thresholds), 1000):
        axs[i, j].text(fpr[k], tpr[k], f"{thresholds[k]:.2f}", fontsize=6, color='black', ha='right', va='bottom')

    # Plot for each pair
    for other_label in label_map.keys():
        if other_label == truth: 
            continue

        mask_pair = (y_test == truth_idx) | (y_test == label_map[other_label])
        y_true_pair = (y_test[mask_pair] == truth_idx).astype(int)
        y_pred_pair = y_pred_probs[mask_pair, truth_idx]
        fpr_pair, tpr_pair, _ = roc_curve(y_true_pair, y_pred_pair)
        auc_pair = roc_auc_score(y_true_pair, y_pred_pair)
        # Respect color scheme for other_label 
        axs[i, j].plot(fpr_pair, tpr_pair, label=f"{other_label} (AUC = {auc_pair:.2f})", ls='--', color=color_map[other_label])

    axs[i,j].set_xlabel("False Positive Rate")
    axs[i,j].set_ylabel("True Positive Rate")
    axs[i,j].set_title(f"{truth} vs X (using Prob({truth}))")
    axs[i,j].legend(loc='lower right')
    axs[i,j].plot([0, 1], [0, 1], alpha=0.5, color='gray')  # Diagonal line for reference

plt.tight_layout()
plt.show()


Which two classes are hardest to distinguish? Can you explain why based on the physics of these processes and the features that we have available for training?

## 1c: Convert the model to an FPGA implementation using hls4ml

For "online" event classification, we need to synthesize the MLP into FPGA firmware, and ensure that it fits within the latency/resource requirements. For this we will use the hls4ml library, which converts a trained Keras model into an FPGA implementation using high-level synthesis (HLS).

Remember, **FPGAs use fixed point arithmetic**. Our network weights, biases and activations will be represented as fixed point numbers. This is different to the floating point representation that is used in your CPU/GPU. We need to be careful when choosing the precision of our fixed point representation, as this can have a big impact on the accuracy of our model. 

You can read more about datatypes here: https://github.com/hlslibs/ac_types/blob/v3.7/pdfdocs/ac_datatypes_ref.pdf

Let's begin by configuring the model conversion. We will use `ac_fixed<16,6>` for all weights, biases and activations. This is sufficiently large to achieve a good accuracy for this simple model.

In [ ]:
hls_config = hls4ml.utils.config_from_keras_model(model, granularity="model", backend='OneAPI')

# We will manually set the precision to ac_fixed<16,6> - this is a standard default choice
# 16 - total number of bits
# 6 - number of bits for the integer part (including the sign bit)
hls_config['Model']['Precision']['default'] = 'ac_fixed<16,6>'

print("-----------------------------------")
print("Configuration")
print(hls_config)
print("-----------------------------------")

What do the different terms mean in the config dict? Lets focus on the important ones for you to understand:
* Precision: speicifies the fixed point precision to use for weights, biases and activations. This choice affects both resource usage and model accuracy. Note the precision can be specified globally (as above) or spearate for each neural network layer.
* Reuse factor: controls hardware reuse, such that 1 means no results i.e. the design uses as many hardware resources as needed to execute all operations in parallel. The lower this factor the faster the inference (lower latency) but more resources are used. This factor can be tuned to meet design contraints (parallelisation).
* Strategy: tells hls4ml to optimize for the lowest possible latency. Other strategies include "Resource", which optimizes for minimal area (however this is not supported by the OneAPI backend).

We will now convert the Keras model into an implementation for an Intel "Agilex7" FPGA. These are a family of high-end FPGAs which are designed for performance-demanding and power-efficient applications, like AI acceleration.

In [ ]:
# Convert the Keras model to an HLS model and write to project directory
proj_dir_name = f"proj_mlp_fixed16-6"

model_hls = hls4ml.converters.convert_from_keras_model(
    model,
    output_dir=proj_dir_name,
    io_type='io_parallel',
    backend='oneAPI',
    part='Agilex7',
    hls_config=hls_config,
    clock_period='1'
)

model_hls.write()

### Emulating the build of the HLS model

In [ ]:
# For linux/Windows: emulate build (this may take ~5 minutes)
# Ignore the segfault
model_hls.build(build_type='report')

In [ ]:
# For macOS: use script to run build inside docker container (this may take ~5 minutes)
# Ignore the segfault 
os.system(f"./build_report.sh {proj_dir_name}")

In [ ]:
# Parse the report from the model dir
report_16_6 = hls4ml.report.oneapi_report.parse_oneapi_report(proj_dir_name)

# Apply the patch and print
hls4ml.report.oneapi_report._make_html_table_template = patched_make_html_table_template
hls4ml.report.oneapi_report.print_oneapi_report(report_16_6)

**So even quite a simple neural network (~3000) trainable parameters can take up a sizeable chunk of the FPGA**.

This raises the questions...
* How can we accelerate more complex ML algorithms without being limited by the resources?
* How can we store many ML algorithms on the same FPGA?

**We need to think smarter in terms of the training (see Part 2)**

### Use HLS model to make predictions and evaluate the performance
Now that we have built the HLS model, we can use it to make predictions on the test set. We need to understand if the HLS model is giving us the same predictions as the original Keras model.

Note, this is not a bit-true emulation, but the OneAPI emulation gives us an idea of how the model will perform on the FPGA. We will compare the predictions from the HLS model to those from the original Keras model and evaluate some of the same performance metrics as before.

In [ ]:
# For linux/Windows: predict with the HLS model (this may take a few minutes)
# First run the compilation
model_hls.compile()

# Extract test data as contiguous array (hls4ml requires this for prediction)
X_test_contig = np.concatenate([test_loader[i][0] for i in range(len(test_loader))])

# Predict with the HLS model
y_hls_pred_probs = model_hls.predict(X_test_contig)
print(f"HLS model predicted probabilities shape: {y_hls_pred_probs.shape}")

In [ ]:
# For macOS: use script to run build inside docker container (this may take ~2 minutes)
# We also need script to have access to the X_test - we will save it in the project directory
# Extract test data as contiguous array (hls4ml requires this for prediction)
X_test_contig = np.concatenate([test_loader[i][0] for i in range(len(test_loader))])
np.save(os.path.join(proj_dir_name, "X_test.npy"), X_test_contig)

# Run script in docker container
os.system(f"./run_in_container.sh \"python evaluate_hls_model.py {proj_dir_name}\"")

# Load saved predictions from project directory (produced with the above script)
y_hls_pred_probs = np.load(os.path.join(proj_dir_name, "y_hls_pred_probs.npy"))
print(f"HLS model predicted probabilities shape: {y_hls_pred_probs.shape}")

# Delete the X_test.npy file from the project directory to save space
os.remove(os.path.join(proj_dir_name, "X_test.npy"))

In [ ]:
# Now compare accuracy of HLS model predictions to original Keras model predictions
y_pred_hls = np.argmax(y_hls_pred_probs, axis=1)
y_pred = np.argmax(model.predict(test_loader), axis=1)

hls_acc_16_6 = np.mean(y_pred_hls == y_test)
keras_acc = np.mean(y_pred == y_test)

print(f"HLS Model Test Accuracy: {hls_acc_16_6:.3f}")
print(f"Keras Model Test Accuracy: {keras_acc:.3f}")

With this precision, our HLS model achieves a very similar accuracy to the original Keras model (with floating point precision). We can also look directly at the output probability distributions for each class.

In [ ]:
# Plot output probability histograms for each class, separated by true class
# Use dashed lines for hls predictions and solid lines for keras predictions, and respect the color scheme for the true classes
fig, axs = plt.subplots(2, 2, figsize=(8, 8))
for pred, pred_idx in label_map.items():

    i = int(np.floor(pred_idx / 2))
    j = int(pred_idx % 2)

    for truth, truth_idx in label_map.items():
        class_mask = (y_test == truth_idx)
        axs[i, j].hist(y_pred_probs[class_mask, pred_idx], bins=50, range=(0,1), histtype='step', label=f"{truth} (Keras)", density=True, color=color_map[list(label_map.keys())[list(label_map.values()).index(truth_idx)]])
        axs[i, j].hist(y_hls_pred_probs[class_mask, pred_idx], bins=50, range=(0,1), histtype='step', label=f"{truth} (HLS)", density=True, color=color_map[list(label_map.keys())[list(label_map.values()).index(truth_idx)]], ls='--')
    axs[i, j].set_xlabel(f"Predicted probability for class: {pred}")
    axs[i, j].set_ylabel("Density")
    axs[i, j].legend(title="True Class")
    axs[i, j].set_xlim(0, 1)
plt.tight_layout()
plt.show()

What do you notice about the probability outputs from the HLS model?

They only take a set of discrete values. This is because the FPGA implementation uses fixed-point arithmetic, which has limited precision compared to the floating-point arithmetic used in the original Keras model. This will lead to some loss of information that can impact the performance of the model. However, we have seen that using <16,6> fixed-point precision allows us to retain a high level of accuracy.

The flip side of this trade-off is that reducing the bit-widths of the network weights/biases will lead to a smaller model size and faster inference time on the FPGA.

## 1d: Accuracy vs resource usage trade-off
Let's investigate this trade-off. First try reducing the bit-widths to <14,6> and see how this impacts the accuracy vs resource usage.

In [ ]:
# Change the fixed-point precision to <14,6> in the hls_config
hls_config['Model']['Precision']['default'] = 'ac_fixed<14,6>'
print("-----------------------------------")
print("Updated Configuration with <14,6> precision")
print(hls_config)
print("-----------------------------------")

# Convert the Keras model to an HLS model and write to project directory
proj_dir_name = f"proj_mlp_fixed14-6"

model_hls = hls4ml.converters.convert_from_keras_model(
    model,
    output_dir=proj_dir_name,
    io_type='io_parallel',
    backend='oneAPI',
    part='Agilex7',
    hls_config=hls_config,
    clock_period='1'
)

model_hls.write()

In [ ]:
# For linux/Windows: emulate build
model_hls.build(build_type='report')

In [ ]:
# For macOS: use script to run build inside docker container
os.system(f"./build_report.sh {proj_dir_name}")

In [ ]:
# Load the reports from both builds and compare resource usage and latency estimates
report_16_6 = hls4ml.report.oneapi_report.parse_oneapi_report("proj_mlp_fixed16-6")
report_14_6 = hls4ml.report.oneapi_report.parse_oneapi_report("proj_mlp_fixed14-6")

# Print both reports with the patch applied
hls4ml.report.oneapi_report._make_html_table_template = patched_make_html_table_template
print("Report for <16,6> precision:")
hls4ml.report.oneapi_report.print_oneapi_report(report_16_6)

print("\n\nReport for <14,6> precision:")
hls4ml.report.oneapi_report.print_oneapi_report(report_14_6)

In [ ]:
# Linux/Windows: predict with the HLS model with <14,6> precision
model_hls.compile()
y_hls_pred_probs_14_6 = model_hls.predict(X_test_contig)

In [ ]:
# MacOS: use script to run build inside docker container and evaluate with <14,6> precision
# First save the X_test_contig in the project directory for access inside the container
np.save(os.path.join(proj_dir_name, "X_test.npy"), X_test_contig)
os.system(f"./run_in_container.sh \"python evaluate_hls_model.py {proj_dir_name}\"")
y_hls_pred_probs_14_6 = np.load(os.path.join(proj_dir_name, "y_hls_pred_probs.npy"))
# Delete the X_test.npy file from the project directory to save space
os.remove(os.path.join(proj_dir_name, "X_test.npy"))

In [ ]:
# Extract accuracy for the <14,6> precision model and compare to the <16,6> model and original Keras model
y_pred_hls_14_6 = np.argmax(y_hls_pred_probs_14_6, axis=1)
hls_acc_14_6 = np.mean(y_pred_hls_14_6 == y_test)
print(f"HLS Model Test Accuracy with <14,6> precision: {hls_acc_14_6:.3f}")
print(f"HLS Model Test Accuracy with <16,6> precision: {hls_acc_16_6:.3f}")
print(f"Keras Model Test Accuracy: {keras_acc:.3f}")

What do you notice about the impact on the accuracy and resource usage?

<div style="background-color:#C2F5DD">

### Exercise: plot resource usage and accuracy for different bit-widths
Repeat the build and evaluation steps for different bit-widths (e.g. <12,6>, <10,6>) and plot the accuracy vs bit-width and resource usage vs bit-width.
* What do you notice about the trade-off between accuracy and resource usage as we reduce the bit-widths?
</div>


In [ ]:
# -- YOUR CODE HERE --

<div style="background-color:#FFE5CC">

### Extension: deep-set architecture
The DeepSet architecture introduces an **inductive bias that respects permutation symmetry** meaning that the output is invariant to the order of the particles in the event.

Each particle is first embedded individually through a shared function $\phi$ (e.g. a small MLP):

$$
h_i = \phi(x_i) \quad i = 1, \ldots, N_{\mathrm{particles}}
$$

These embeddings are then **aggregated using a permutation-invariant operation** such as summation, mean, or max:

$$
H = \rho\left(\sum_{i=1}^{N_{\mathrm{particles}}} h_i\right)
$$

where $\rho$ is another function (e.g. another small MLP) that maps the aggregated embedding to the output e.g., class probabilities.

Try implementing a DeepSet architecture for this problem and compare its performance and FPGA implementation to the MLP. We will help you out by providing the data loaders and the model architecture. You will then need to train the model and convert it to an FPGA implementation using hls4ml. You may need to modify the hls4ml configuration to meet the timing requirements of the FPGA.
</div>

In [ ]:
# Initialise the data loaders
# Key difference: we will not flatten the data
flatten = False

# We will again use a batch size of 512
train_loader_deepset = ParticleDataLoader(X_train, y_train, batch_size=512, flatten=flatten)
test_loader_deepset = ParticleDataLoader(X_test, y_test, batch_size=512, flatten=flatten)
print(f"DataLoader gives input shape: {train_loader_deepset[0][0].shape} and labels shape: {train_loader_deepset[0][1].shape}")

In [ ]:
# Define DeepSet model architecture
input_shape = X_train.shape[1:]  # This will be (num_particles, features) = (16, 3)
inputs = layers.Input(shape=input_shape)

# Shared particle embedding network (phi) - each particle is independently embedded by an MLP
x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(inputs)
x = layers.Dropout(0.1)(x)
x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)

# Permutation-invariant aggregation 
x = layers.GlobalAveragePooling1D()(x)

# Final classification MLP (rho)
x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
outputs = layers.Dense(4, activation='softmax')(x)
model_deepset = models.Model(inputs=inputs, outputs=outputs)

In [ ]:
# -- YOUR CODE HERE --

# Part 2: High granularity quantization
Major downfall in the previous approach was:
* Neural network was trained with floating point precision
* We then quantized the model to fixed-point precision for the FPGA implementation
* This can lead to a significant drop in accuracy, especially for lower bit-widths.

In other words, the neural network training was unaware of the eventual deployment on the FPGA, and the quantization was done as an afterthought. 

In this part of the tutorial we will explore how to train the neural network with **quantization-aware training**. In this way, the model will learn to be robust to the quantization effects, which will help to retain a higher level of accuracy when we convert to the FPGA implementation.

A few years ago, the standard approach for quantization-aware training was to fix the bit-widths of the neural network weights and activations during training using QKeras. The quantization was applied during the forward pass, but the gradients were computed using the straight-through estimator (STE) to allow for backpropagation. However, this approach is limited...
* It does not allow for any flexibility in the quantization scheme during training. This presents a huge search space when we want to explore different bit-widths for the network weights and activations e.g. uniform vs mixed precision, different bit-widths for different layers, etc.
* Pruning (another method for reducing the model footprint) must be applied on top of the QKeras training.

Both of these issues have now been overcome with the **High granularity quantization (hgq)** library. 
* Bit-widths are defined per network parameter (weights, biases, activations)
* These bit-widths are automatically optimized during training by gradient descent (see sections 3.1 and 3.2 of https://arxiv.org/abs/2405.00645)
* Accurately estimate resource usage using Effective Bit Operations (EBOPs) which enters as a differentiable term in the loss function. You can think of it as a regularization term that encourages lower bit-widths.
* Pruning automatically occurs when bit-widths reach zero.

This package is directly integrated with hls4ml... so let's try it out and see how it performs on our dataset.

In [ ]:
# HGQ imports
import hgq
from hgq.config import LayerConfigScope, QuantizerConfigScope
from hgq.layers import QDense
from hgq.utils.sugar import FreeEBOPs, PBar
from hgq.regularizers import MonoL1
from hgq.utils.sugar import BetaScheduler, FreeEBOPs, PBar

from keras.callbacks import ReduceLROnPlateau, TerminateOnNaN

We will schedule the "beta" term (i.e. the regularization strength for the EBOPs) to gradually increase during training. This allows us to explore the trade-off between model performance (accuracy) and resource usage throughout the training process.

In [ ]:
N_epochs_hgq = 100
beta_initial = 1e-8
beta_final = 1e-4
beta_sched = lambda x: beta_final * np.linspace(0, 1, N_epochs_hgq)[x]**3 + beta_initial
beta_scheduler = BetaScheduler(beta_sched)

# Make a plot of the beta schedule to check it looks sensible
fig, ax = plt.subplots(1, 2, figsize=(10,4))
x = np.arange(N_epochs_hgq)
y = beta_sched(x)
ax[0].plot(x, y)
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Beta value")
ax[1].plot(x, y)
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Beta value")
ax[1].set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Build the quantized MLP model using HGQ
# Different syntax to what we were using before
def get_hgq_model(input_shape, layer_dim, num_classes, beta_initial):
    # Quantization scope for setting default quantization type and overflow mode
    # Second scope overrides the first for the datalane (activations) place, allowing us to have different quantization configs for weights and activations
    qscope0 = QuantizerConfigScope(k0=1, b0=8, i0=2, br=MonoL1(1e-8), overflow_mode='WRAP')
    qscope1 = QuantizerConfigScope(place='datalane', k0=1, f0=8, fr=MonoL1(1e-9), ir=MonoL1(1e-8))
    # Layer scopes define which layers the quantization configs apply: default = all
    lscope = LayerConfigScope(beta0=beta_initial)

    with lscope, qscope0, qscope1:
        inputs = layers.Input(shape=(input_shape,))
        x = QDense(layer_dim, activation='relu')(inputs)
        x = QDense(layer_dim, activation='relu')(x)
        x = QDense(layer_dim, activation='relu')(x)
        # Note the lack of softmax activation here - the loss function expects logits and this allows for better quantization of the output layer
        outputs = QDense(num_classes)(x)

    return models.Model(inputs=inputs, outputs=outputs)

In [ ]:
input_shape = train_loader[0][0].shape[1]
layer_dim = 32
num_classes = len(label_map)
model_hgq = get_hgq_model(input_shape, layer_dim, num_classes, beta_initial)

In [ ]:
# Model compile with HGQ
# Track loss, accuracy and ebops during training (also terminate if NaNs are encountered in the loss)
pbar = PBar('loss: {loss:.3f}/{val_loss:.3f} - acc: {accuracy:.3f}/{val_accuracy:.3f}')
ebops_tracker = FreeEBOPs()
nan_terminate = TerminateOnNaN()

# Add ModelCheckpoint callback
# Now save model for every epoch as we navigate the Pareto-front of accuracy vs EBOPs during training
checkpoint_path = "models/hgq_epoch_{epoch:02d}.keras"
checkpoint_cb = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_accuracy',
    save_best_only=False,  # Save every epoch
    save_weights_only=False,
    save_freq='epoch',
    verbose=1
)

# Define loss
loss_logits = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# Define optimizer
optimizer = Adam()

model_hgq.compile(
    optimizer=optimizer, 
    loss=loss_logits,
    metrics=['accuracy'],
    jit_compile=True,
    steps_per_execution=4
)

model_hgq.summary()

In [ ]:
# Model training
callbacks = [ebops_tracker, beta_scheduler, nan_terminate, checkpoint_cb]
history_hgq = model_hgq.fit(
    train_loader,
    validation_data=test_loader,
    epochs=N_epochs_hgq,
    batch_size=512,
    callbacks=callbacks,
    verbose=1
)

The model has trained! 

Let's look at how the accuracy and EBOPs evolve during training...

In [ ]:
# Make a scatter plot of EBOPs vs epoch and Accuracy vs epoch
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(np.arange(N_epochs_hgq), history_hgq.history['ebops'], label='EBOPs')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('EBOPs')
ax[0].set_yscale('log')
ax[0].legend()

ax[1].plot(np.arange(N_epochs_hgq), history_hgq.history['accuracy'], label='Train Accuracy', marker='o', ls='None')
ax[1].plot(np.arange(N_epochs_hgq), history_hgq.history['val_accuracy'], label='Test Accuracy', marker='o', ls='None')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Accuracy')
ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot distribution of bitwidths - different histogram for different epochs
epochs = [1, 10, 20, 30, 40, 60, 80]

fig, ax = plt.subplots(figsize=(6, 4))

for epoch in epochs:
    model_epoch = keras.models.load_model(f"models/hgq_epoch_{epoch:02d}.keras")
    bitwidths = []
    for layer in model_epoch.layers:
        if isinstance(layer, QDense):
            bitwidths += list(layer.kq.bits.flatten())
            bitwidths += list(layer.iq.bits.flatten())
            bitwidths += list(layer.bq.bits.flatten())

    # Plot histogram of bitwidths for this epoch
    ax.hist(bitwidths, bins=11, range=(-0.5, 10.5), histtype='step', label=f'HGQ model: epoch {epoch}')

ax.set_xlabel('Bit Width')
ax.set_ylabel('Count')
ax.set_xticks(range(0, 11))
ax.legend()
plt.tight_layout()
plt.show()

We are traversing the Pareto frontier of accuracy vs resource usage during training. Lets plot this directly...

In [ ]:
# Plot the Pareto front of Accuracy vs EBOPs
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(history_hgq.history['ebops'], history_hgq.history['val_accuracy'], label='HGQ Models', c=np.arange(N_epochs_hgq), cmap='viridis', marker='o')
ax.set_xlabel('EBOPs')
ax.set_ylabel('Test Accuracy')
ax.set_xscale('log')
ax.set_ylim(0.5,0.6)
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()  

It is up to us to decide which model along the Pareto frontier is the best choice for our application. This depends on the specific requirements in terms of resources/latency vs accuracy.

Let's choose a model and synthesize with hls4ml to examine the resource usage.

In [ ]:
# Let's pick the model at epoch 40 - at this point we are still obtaining a high accuracy with a much reduced EBOP count
model_hgq_epoch_40 = keras.models.load_model("models/hgq_epoch_40.keras", compile=False)

# Evaluate the model on the test set
y_pred_logits_hgq = model_hgq_epoch_40.predict(test_loader)
# Obtain predicted classes
y_pred_hgq = np.argmax(y_pred_logits_hgq, axis=1)
hgq_acc = np.mean(y_pred_hgq == y_test)
print(f"HGQ Model Test Accuracy at Epoch 40: {hgq_acc:.3f}")

In [ ]:
# Now we will do the hls4ml conversion
# Bitwidths are extracted automatically from the HGQ model (2 bits acts as a placeholder)
hls_config_hgq = {
    'Model': {
        'Precision': 'ac_fixed<2,0>',
        'ReuseFactor': 1
    }
}

proj_dir_name = f"proj_hgq_epoch_40"

model_hls_hgq = hls4ml.converters.convert_from_keras_model(
    model_hgq_epoch_40,
    output_dir=proj_dir_name,
    io_type='io_parallel',
    backend='oneAPI',
    part='Agilex7',
    hls_config=hls_config_hgq,
    clock_period='1'
)
model_hls_hgq.write()

In [ ]:
# For linux/Windows: emulate build
model_hls_hgq.build(build_type='report')

In [ ]:
# For macOS: use script to run build inside docker container
os.system(f"./build_report.sh {proj_dir_name}")

In [ ]:
# Extract report and print
report_hgq_epoch_40 = hls4ml.report.oneapi_report.parse_oneapi_report(proj_dir_name)
hls4ml.report.oneapi_report._make_html_table_template = patched_make_html_table_template
print(f"HGQ Model Report at Epoch 40:")
hls4ml.report.oneapi_report.print_oneapi_report(report_hgq_epoch_40)

**We are achieving much lower resources!** Now let's check the accuracy of the HLS model...

In [ ]:
# Linux/Windows: predict with the HLS model
model_hls_hgq.compile()
y_hgq_hls_pred_logits = model_hls_hgq.predict(X_test_contig)

In [ ]:
# MacOS: use script to predict with the HLS model inside docker container
# First save the X_test_contig in the project directory for access inside the container
np.save(os.path.join(proj_dir_name, "X_test.npy"), X_test_contig)
os.system(f"./run_in_container.sh \"python evaluate_hls_model.py {proj_dir_name}\"")
y_hgq_hls_pred_logits = np.load(os.path.join(proj_dir_name, "y_hls_pred_probs.npy"))
# Delete the X_test.npy file from the project directory to save space
os.remove(os.path.join(proj_dir_name, "X_test.npy"))

In [ ]:
# Extract the accuracy of the HGQ HLS model and compare to the original HGQ Keras model
y_hgq_hls_pred = np.argmax(y_hgq_hls_pred_logits, axis=1)
hgq_hls_acc = np.mean(y_hgq_hls_pred == y_test)
print(f"HGQ HLS Model Test Accuracy at Epoch 40: {hgq_hls_acc:.3f}")
print(f"HGQ Keras Model Test Accuracy at Epoch 40: {hgq_acc:.3f}")

**We are able to achieve the same level of accuracy as the original Keras model with a much smaller resource usage on the FPGA.**

This is because the quantization has been optimized during the training process!

<div style="background-color:#C2F5DD">

### Exercise: comparing to post-training quantization 
Add the HGQ results to the plots you obtained in the exercise at the end of Part 1.
</div>

In [ ]:
# -- YOUR CODE HERE --

<div style="background-color:#C2F5DD">

### Exercise: investigating EBOPs
So is the EBOPs metric in the loss term representative of the actual resource usage on the FPGA? 

Try looping through a few HGQ models at different points on the Pareto frontier (different EBOPs). Create a scatter plot of LUT/FF usage vs EBOPs.
</div>

In [ ]:
# -- YOUR CODE HERE --